# Time-Resolved XRD Analysis

Load compact processed 1-D stacks into xarray, keep raw/cake rows lazy,
normalize and bin with provenance, inspect a pilot fit, then derive a
lattice and explicitly illustrative temperature history. Physical rates
require explicit seconds and a calibration.


In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from xrd_tools.analysis import (
    LinearThermalExpansion, PeakFitPlan, add_lattice_results,
    add_temperature_results, bin_time_resolved, fit_peak_series,
    export_time_resolved_results,
    flag_fit_quality, flag_normalization_outliers, load_time_resolved_series,
    normalize_monitor, normalize_reference_band, select_time_zero,
)
from xrd_tools.gui.widgets import ImageViewer, PatternViewer
from xrd_tools.viz import plot_peak_fit_frame, plot_thermal_history, plot_time_resolved_waterfall


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
processed_selection = TEST_DATA / "Pt_test_burst_00007.nxs"
source_root = TEST_DATA if TEST_DATA.exists() else None
q_band = widgets.FloatRangeSlider(value=(3.05, 3.20), min=2.4, max=3.3, step=0.01, description="reference q", continuous_update=False)
bin_size = widgets.BoundedIntText(value=2, min=1, max=100, description="bin size")
pilot_button = widgets.Button(description="Run pilot fit", button_style="primary")
batch_button = widgets.Button(description="Run bounded batch", button_style="primary")
export_enabled = widgets.Checkbox(value=False, description="Enable compact export")
export_directory = Path(os.environ.get("XDART_NOTEBOOK_OUTPUT", tempfile.gettempdir()))
status = widgets.HTML("<i>Buttons run expensive fits; display controls only redraw saved data.</i>")
display(widgets.VBox([q_band, bin_size, widgets.HBox([pilot_button, batch_button]), export_enabled, status]))


In [ ]:
if SMOKE_MODE:
    smoke_file = Path(tempfile.gettempdir()) / "xdart_notebook_time_resolved_smoke.nxs"
    q = np.linspace(2.45, 3.30, 240)
    frames = np.arange(16, dtype=np.int64)
    centers = 2.765 - 0.0008 * frames
    stack = np.array([15 + 180 * np.exp(-0.5 * ((q - center) / 0.018) ** 2) for center in centers], dtype=np.float32)
    with h5py.File(smoke_file, "w") as h5:
        entry = h5.create_group("entry")
        one_d = entry.create_group("integrated_1d")
        one_d.create_dataset("frame_index", data=frames)
        q_dataset = one_d.create_dataset("q", data=q)
        q_dataset.attrs["units"] = "q_A^-1"
        one_d.create_dataset("intensity", data=stack)
        one_d.create_dataset("sigma", data=np.sqrt(stack))
        scan_data = entry.create_group("scan_data")
        scan_data.create_dataset("frame_index", data=frames)
        scan_data.create_dataset("elapsed_time", data=frames * 0.002)
        scan_data.create_dataset("i0", data=np.linspace(0.98, 1.02, len(frames)))
        cakes = entry.create_group("integrated_2d")
        cakes.create_dataset("frame_index", data=frames)
        cakes.create_dataset("q", data=q)
        cakes.create_dataset("chi", data=np.linspace(-30, 30, 12))
        cakes.create_dataset("intensity", data=np.broadcast_to(stack[:, None, :], (len(frames), 12, len(q))))
        frame_groups = entry.create_group("frames")
        for frame in frames:
            frame_groups.create_group(f"frame_{frame:04d}").create_dataset("thumbnail", data=np.ones((8, 8), dtype=np.uint8))
    selected_paths = smoke_file
else:
    assert processed_selection.exists(), f"Missing processed NeXus: {processed_selection}"
    selected_paths = processed_selection

series = load_time_resolved_series(selected_paths, metadata_keys=("i0",), source_root=source_root)
raw_status = "not requested"
thumbnail = series.get_thumbnail(0)
cake = series.get_cake(0)
try:
    raw = series.get_raw(0)
    raw_status = f"raw shape={raw.shape}"
except KeyError:
    raw_status = "raw source unavailable; thumbnail remains distinct"
{"patterns": series.dataset.sizes["pattern"], "q_points": series.dataset.sizes["q"], "raw": raw_status, "cake_shape": cake.intensity.shape}


In [ ]:
normalized = normalize_monitor(series.dataset, "i0")
normalized = normalize_reference_band(normalized, q_range=tuple(q_band.value), intensity_var="intensity_normalized", output_var="intensity_band_normalized")
flagged = flag_normalization_outliers(normalized)
binned = bin_time_resolved(flagged, bin_size=bin_size.value)
zeroed = select_time_zero(binned, zero_pattern=0)
waterfall = plot_time_resolved_waterfall(zeroed, intensity_var="intensity_band_normalized", log_intensity=True)
pattern_viewer = PatternViewer(patterns=[(zeroed.q.values, zeroed.intensity_band_normalized.isel(pattern=0).values, "row 0")])
image_viewer = ImageViewer(thumbnail, title="Stored thumbnail")
display(widgets.HBox([pattern_viewer.widget, image_viewer.widget]))
waterfall


In [ ]:
plan = PeakFitPlan(positions=(2.76,), model="gaussian", background="linear", sigma_init=0.02, center_bounds_delta=0.08)
fits = fit_peak_series(zeroed, plan, intensity_var="intensity_band_normalized", q_range=(2.60, 2.95), pattern_indices=np.arange(min(8, zeroed.sizes["pattern"])))
fits = flag_fit_quality(fits, max_center_error=0.02)
lattice = add_lattice_results(fits, hkls=((1, 1, 1),))
calibration = LinearThermalExpansion(float(lattice.lattice_mean_A.isel(fit_pattern=0)), 300.0, 9e-6)
thermal = add_temperature_results(lattice, calibration, time_coord="time")
if export_enabled.value:
    export_time_resolved_results(
        thermal,
        netcdf_path=export_directory / "time_resolved_results.nc",
        csv_path=export_directory / "time_resolved_scalars.csv",
    )
display(plot_peak_fit_frame(thermal, 0))
display(plot_thermal_history(thermal))
{"fit_rows": thermal.sizes["fit_pattern"], "valid": int(thermal.fit_valid.sum()), "temperature_units": thermal.temperature_K.attrs["units"]}
